# Day 6 (Saturday): Statistics for ML
## Week 1 — Python for ML & Math Refresh

**Time:** 3 hours  
**Goal:** Refresh the statistical concepts that underpin ML: distributions, hypothesis testing, Bayes theorem, and the metrics you'll use to evaluate every model.

> 💡 **As a Data Engineer, you've seen data quality issues, skewed distributions, and missing values across client projects. Statistics gives you the vocabulary and tools to reason about them formally.**

---

## 1. Descriptive Statistics — Summarising Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# ── Measures of Central Tendency ──
data = np.array([45, 52, 48, 61, 55, 42, 58, 49, 53, 200])  # Note the outlier!

mean = np.mean(data)
median = np.median(data)
mode = stats.mode(data, keepdims=True).mode[0]

print(f"Data: {data}")
print(f"Mean:   {mean:.1f}")    # Sensitive to outliers
print(f"Median: {median:.1f}")  # Robust to outliers
print(f"Mode:   {mode}")

# 💡 When mean >> median → right-skewed (outliers pulling mean up)
# 💡 Use median for salary data, house prices — always check for skew

In [ ]:
# ── Measures of Spread ──
clean_data = np.array([45, 52, 48, 61, 55, 42, 58, 49, 53, 47])

variance = np.var(clean_data)
std_dev = np.std(clean_data)
iqr = np.percentile(clean_data, 75) - np.percentile(clean_data, 25)
data_range = clean_data.max() - clean_data.min()

print(f"Data: {clean_data}")
print(f"Variance: {variance:.2f}")
print(f"Std Dev:  {std_dev:.2f}")
print(f"IQR:      {iqr:.2f}")
print(f"Range:    {data_range}")

# ── Outlier detection with IQR ──
Q1 = np.percentile(clean_data, 25)
Q3 = np.percentile(clean_data, 75)
lower_bound = Q1 - 1.5 * iqr
upper_bound = Q3 + 1.5 * iqr
print(f"\nOutlier bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"Outliers: {clean_data[(clean_data < lower_bound) | (clean_data > upper_bound)]}")

## 2. Probability Distributions

In [ ]:
# ── Normal (Gaussian) Distribution ──
# The most important distribution in ML — Central Limit Theorem guarantees it shows up everywhere

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Standard normal
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x, 0, 1)
axes[0, 0].plot(x, y, 'b-', linewidth=2)
axes[0, 0].fill_between(x, y, where=(x >= -1) & (x <= 1), alpha=0.3, color='blue', label='68%')
axes[0, 0].fill_between(x, y, where=(x >= -2) & (x <= 2), alpha=0.15, color='green', label='95%')
axes[0, 0].set_title('Normal Distribution', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].annotate('68% within 1σ', xy=(0, 0.2), fontsize=10, ha='center')

# Different means and std devs
for mu, sigma, color in [(0, 1, 'blue'), (2, 1, 'red'), (0, 2, 'green')]:
    y = stats.norm.pdf(x, mu, sigma)
    axes[0, 1].plot(x, y, color=color, linewidth=2, label=f'μ={mu}, σ={sigma}')
axes[0, 1].set_title('Effect of μ and σ', fontweight='bold')
axes[0, 1].legend()

# Uniform distribution
x_u = np.linspace(-1, 3, 1000)
y_u = stats.uniform.pdf(x_u, 0, 2)  # [0, 2]
axes[1, 0].plot(x_u, y_u, 'r-', linewidth=2)
axes[1, 0].fill_between(x_u, y_u, alpha=0.3, color='red')
axes[1, 0].set_title('Uniform Distribution [0, 2]', fontweight='bold')

# Binomial distribution
n, p = 20, 0.3
x_b = np.arange(0, 21)
y_b = stats.binom.pmf(x_b, n, p)
axes[1, 1].bar(x_b, y_b, color='purple', alpha=0.7)
axes[1, 1].set_title(f'Binomial (n={n}, p={p})', fontweight='bold')

plt.tight_layout()
plt.savefig('distributions.png', dpi=150)
plt.show()

## 3. Probability & Bayes Theorem

In [ ]:
# ── Bayes Theorem: P(A|B) = P(B|A) × P(A) / P(B) ──
# "Update your belief given new evidence"

# Example: Spam classification
# P(spam) = 0.3 (30% of emails are spam)
# P(contains "free" | spam) = 0.8 (80% of spam says "free")
# P(contains "free" | not spam) = 0.1 (10% of legit emails say "free")

# What's P(spam | contains "free")?

p_spam = 0.3
p_free_given_spam = 0.8
p_free_given_not_spam = 0.1

# P(free) = P(free|spam)×P(spam) + P(free|not spam)×P(not spam)
p_free = p_free_given_spam * p_spam + p_free_given_not_spam * (1 - p_spam)

# Bayes theorem
p_spam_given_free = (p_free_given_spam * p_spam) / p_free

print(f"P(spam) = {p_spam}")
print(f"P('free' | spam) = {p_free_given_spam}")
print(f"P('free' | not spam) = {p_free_given_not_spam}")
print(f"P('free') = {p_free:.3f}")
print(f"\nP(spam | 'free') = {p_spam_given_free:.3f}")
print(f"\n→ If an email contains 'free', there's a {p_spam_given_free*100:.1f}% chance it's spam")

# 💡 This is EXACTLY how Naive Bayes classifier works
# 💡 Also the foundation of Bayesian inference in ML

## 4. Correlation vs Causation

In [ ]:
# ── Pearson Correlation: linear relationship strength ──
np.random.seed(42)
n = 100

# Strong positive correlation
x1 = np.random.randn(n)
y1 = 2 * x1 + np.random.randn(n) * 0.5

# No correlation
x2 = np.random.randn(n)
y2 = np.random.randn(n)

# Non-linear relationship (correlation misses this!)
x3 = np.linspace(-3, 3, n)
y3 = x3**2 + np.random.randn(n) * 0.5

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, x, y, title in [(axes[0], x1, y1, 'Strong Linear'),
                          (axes[1], x2, y2, 'No Correlation'),
                          (axes[2], x3, y3, 'Non-linear')]:
    ax.scatter(x, y, alpha=0.5, s=30)
    r = np.corrcoef(x, y)[0, 1]
    ax.set_title(f'{title}\nr = {r:.3f}', fontweight='bold')

plt.tight_layout()
plt.savefig('correlation_types.png', dpi=150)
plt.show()

# 💡 Pearson r ≈ 0 does NOT mean no relationship — check scatter plots!
# 💡 Use Spearman for non-linear monotonic relationships

## 5. Hypothesis Testing — Making Decisions from Data

In [ ]:
# ── t-test: "Is there a real difference between two groups?" ──
# Example: Do engineers earn more than marketers?

np.random.seed(42)
engineers_salary = np.random.normal(95000, 15000, 50)
marketers_salary = np.random.normal(85000, 12000, 50)

t_stat, p_value = stats.ttest_ind(engineers_salary, marketers_salary)

print(f"Engineers mean: £{engineers_salary.mean():,.0f}")
print(f"Marketers mean: £{marketers_salary.mean():,.0f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.6f}")
print(f"\nSignificant at α=0.05? {'YES' if p_value < 0.05 else 'NO'}")

# 💡 p-value < 0.05 → reject null hypothesis → difference is statistically significant
# 💡 In ML: used to compare model performances, A/B testing

In [ ]:
# ── Chi-squared test: are categorical variables related? ──
# Example: Is department related to performance rating?

observed = np.array([[30, 45, 25],    # Engineering: Low, Med, High
                     [40, 35, 25],    # Marketing
                     [20, 50, 30]])   # Sales

chi2, p_val, dof, expected = stats.chi2_contingency(observed)
print(f"Chi-squared: {chi2:.3f}")
print(f"p-value: {p_val:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"\nExpected frequencies:\n{expected.round(1)}")
print(f"\nSignificant? {'YES' if p_val < 0.05 else 'NO'}")

## 6. Central Limit Theorem (CLT) — Why Normal Distribution is Everywhere

In [ ]:
# ── CLT: Sample means follow a normal distribution, regardless of the population ──
np.random.seed(42)

# Start with a VERY non-normal distribution (exponential)
population = np.random.exponential(scale=5, size=100000)

# Take many samples of different sizes and compute their means
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, sample_size in enumerate([1, 5, 30, 100]):
    sample_means = [np.random.choice(population, sample_size).mean() for _ in range(1000)]
    axes[i].hist(sample_means, bins=30, density=True, alpha=0.7, color='#2E75B6')
    axes[i].set_title(f'n = {sample_size}', fontweight='bold')
    axes[i].set_xlim(0, 15)

plt.suptitle('Central Limit Theorem: Sample Means Become Normal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('clt.png', dpi=150)
plt.show()

# 💡 Even from a skewed population, means of 30+ samples look normal
# 💡 This is why many ML assumptions about "normally distributed errors" work

## 7. Information Theory Basics — Used in Decision Trees

In [ ]:
# ── Entropy: measure of "disorder" or "uncertainty" ──
# High entropy = unpredictable, Low entropy = predictable

def entropy(probs):
    probs = np.array(probs)
    probs = probs[probs > 0]  # avoid log(0)
    return -np.sum(probs * np.log2(probs))

# Fair coin: maximum uncertainty
print(f"Fair coin [0.5, 0.5]: {entropy([0.5, 0.5]):.4f} bits")

# Biased coin: more predictable
print(f"Biased coin [0.9, 0.1]: {entropy([0.9, 0.1]):.4f} bits")

# Certain outcome: no uncertainty
print(f"Certain [1.0, 0.0]: {entropy([1.0, 0.0]):.4f} bits")

# 3-class problem
print(f"\nEqual 3-class [0.33, 0.33, 0.33]: {entropy([1/3, 1/3, 1/3]):.4f} bits")
print(f"Dominant class [0.8, 0.1, 0.1]: {entropy([0.8, 0.1, 0.1]):.4f} bits")

# 💡 Decision trees split to MINIMIZE entropy (maximize information gain)
# 💡 Cross-entropy loss in neural networks is based on this concept

## 8. Key Statistics → ML Mapping

| Statistical Concept | ML Application |
|---------------------|----------------|
| Mean, Median, Std | Feature scaling, data understanding |
| Normal Distribution | Weight initialisation, error assumptions |
| Bayes Theorem | Naive Bayes classifier, Bayesian inference |
| Correlation | Feature selection, multicollinearity |
| Hypothesis testing | Model comparison, A/B testing |
| Central Limit Theorem | Why batch statistics work (BatchNorm) |
| Entropy | Decision tree splits, cross-entropy loss |
| Percentiles/IQR | Outlier detection, data cleaning |

---
**✅ Day 6 Complete!** Tomorrow: Mini-project with real data (Kaggle EDA).